# S08-demo-01 — Честное сравнение моделей в едином протоколе

Этот ноутбук показывает, как **сравнивать несколько моделей корректно и воспроизводимо**.

Логика семинара:

- у всех моделей один и тот же `train/test split`;
- подбор гиперпараметров идёт через CV **только на train**;
- test используется только для честной итоговой оценки;
- результат фиксируется артефактами.

Сравниваем три семейства моделей:

- **Logistic Regression** — сильный линейный baseline;
- **Random Forest** — ансамбль деревьев;
- **HistGradientBoosting** — современный бустинг по табличным данным.

Цель ноутбука — не «выжать максимум», а показать **правильный инженерный протокол сравнения**.

## 0. Настройки, импорты, пути для артефактов

Сразу фиксируем `RANDOM_STATE`, создаём папку `artifacts/` и подключаем библиотеки для обучения, метрик, графиков и сохранения модели.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
ARTIFACTS_DIR = Path.cwd() / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 200)

print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")
print(f"RANDOM_STATE = {RANDOM_STATE}")

In [ ]:
def compute_metrics(y_true, y_pred, y_score=None):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": None,
    }
    if y_score is not None:
        try:
            metrics["roc_auc"] = float(roc_auc_score(y_true, y_score))
        except Exception:
            metrics["roc_auc"] = None
    return metrics

def append_result(results, model_name, split_name, y_true, y_pred, y_score=None, extra=None):
    row = {"model": model_name, "split": split_name}
    row.update(compute_metrics(y_true, y_pred, y_score))
    if extra:
        row.update(extra)
    results.append(row)

def run_grid_search(name, estimator, param_grid, X_train, y_train, X_test, y_test, cv, scoring="roc_auc"):
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    y_train_score = None
    y_test_score = None

    if hasattr(best_model, "predict_proba"):
        y_train_score = best_model.predict_proba(X_train)[:, 1]
        y_test_score = best_model.predict_proba(X_test)[:, 1]
    elif hasattr(best_model, "decision_function"):
        y_train_score = best_model.decision_function(X_train)
        y_test_score = best_model.decision_function(X_test)

    append_result(
        results=results,
        model_name=name,
        split_name="train",
        y_true=y_train,
        y_pred=y_train_pred,
        y_score=y_train_score,
        extra={"best_cv_score": float(search.best_score_), "best_params": search.best_params_},
    )
    append_result(
        results=results,
        model_name=name,
        split_name="test",
        y_true=y_test,
        y_pred=y_test_pred,
        y_score=y_test_score,
        extra={"best_cv_score": float(search.best_score_), "best_params": search.best_params_},
    )

    return search, best_model

def select_best_row(df):
    df_test = df[df["split"] == "test"].copy()
    if df_test["roc_auc"].notna().any():
        df_test = df_test.sort_values(by=["roc_auc", "f1", "accuracy"], ascending=False, na_position="last")
    else:
        df_test = df_test.sort_values(by=["f1", "accuracy"], ascending=False)
    return df_test.iloc[0]

## 1. Данные и фиксированное разбиение train/test

Возьмём `breast_cancer` из `sklearn` — это табличная задача бинарной классификации.

Самое важное здесь — **split фиксируется один раз и не меняется**.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X shape:", X.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print()
print("Target distribution (full):")
print(y.value_counts(normalize=True).sort_index())

X.head()

## 2. Baseline’ы

Перед сравнением содержательных моделей полезно иметь две точки отсчёта:

- `DummyMostFrequent` — всегда предсказывает самый частый класс;
- `DummyStratified` — случайные предсказания с сохранением частот классов.

In [ ]:
results = []
model_store = {}
search_store = {}

dummy_models = {
    "DummyMostFrequent": DummyClassifier(strategy="most_frequent"),
    "DummyStratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}

for name, model in dummy_models.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_score = None
    y_test_score = None
    if hasattr(model, "predict_proba"):
        y_train_score = model.predict_proba(X_train)[:, 1]
        y_test_score = model.predict_proba(X_test)[:, 1]

    append_result(results, name, "train", y_train, y_train_pred, y_train_score, extra={"best_cv_score": None, "best_params": None})
    append_result(results, name, "test", y_test, y_test_pred, y_test_score, extra={"best_cv_score": None, "best_params": None})

pd.DataFrame(results)[lambda d: d["split"] == "test"].sort_values(by=["accuracy", "f1"], ascending=False)

## 3. Кандидаты: логистическая регрессия, лес, бустинг

Для всех моделей используем **одинаковый CV на train** и общий критерий `roc_auc`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = "roc_auc"

print(cv)
print("Scoring:", SCORING)

### 3.1 Logistic Regression

Для логистической регрессии preprocessing обязателен:

- `SimpleImputer`;
- `StandardScaler`;
- затем классификатор.

Даже если пропусков в данных нет, это правильный инженерный шаблон.

In [ ]:
logreg_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)),
    ]
)

logreg_grid = {
    "model__C": [0.1, 1.0, 3.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

logreg_search, logreg_best = run_grid_search(
    name="LogisticRegression",
    estimator=logreg_pipeline,
    param_grid=logreg_grid,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=cv,
    scoring=SCORING,
)

search_store["LogisticRegression"] = logreg_search
model_store["LogisticRegression"] = logreg_best

print("Best CV score:", round(logreg_search.best_score_, 4))
print("Best params:", logreg_search.best_params_)

### 3.2 Random Forest

Для леса масштабирование не требуется. Подбираем несколько разумных параметров:
`n_estimators`, `max_depth`, `min_samples_leaf`, `class_weight`.

In [ ]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 4, 8],
    "min_samples_leaf": [1, 3, 5],
    "class_weight": [None, "balanced"],
}

rf_search, rf_best = run_grid_search(
    name="RandomForest",
    estimator=rf_model,
    param_grid=rf_grid,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=cv,
    scoring=SCORING,
)

search_store["RandomForest"] = rf_search
model_store["RandomForest"] = rf_best

print("Best CV score:", round(rf_search.best_score_, 4))
print("Best params:", rf_search.best_params_)

### 3.3 HistGradientBoosting

`HistGradientBoostingClassifier` — представитель boosting-подхода для табличных числовых данных.

In [ ]:
hgb_model = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

hgb_grid = {
    "learning_rate": [0.03, 0.1],
    "max_depth": [None, 3, 5],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [10, 20],
}

hgb_search, hgb_best = run_grid_search(
    name="HistGradientBoosting",
    estimator=hgb_model,
    param_grid=hgb_grid,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    cv=cv,
    scoring=SCORING,
)

search_store["HistGradientBoosting"] = hgb_search
model_store["HistGradientBoosting"] = hgb_best

print("Best CV score:", round(hgb_search.best_score_, 4))
print("Best params:", hgb_search.best_params_)

## 4. Сводка результатов

Сначала сравниваем модели по `ROC-AUC`, затем по `F1` и `accuracy`.

In [ ]:
results_df = pd.DataFrame(results)

display(results_df.sort_values(by=["model", "split"]).reset_index(drop=True))

test_summary = (
    results_df[results_df["split"] == "test"]
    .copy()
    .sort_values(by=["roc_auc", "f1", "accuracy"], ascending=False, na_position="last")
    .reset_index(drop=True)
)

print("Итоговая сводка по test:")
display(test_summary[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "best_cv_score", "best_params"]])

## 5. Сохраняем результаты сравнения

После эксперимента должны остаться:

- `metrics_all_splits.json`;
- `metrics_test.csv`;
- `search_summary.json`.

In [ ]:
search_summary = {}
for model_name, search in search_store.items():
    search_summary[model_name] = {
        "best_score_cv": float(search.best_score_),
        "best_params": search.best_params_,
        "scoring": SCORING,
        "cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    }

metrics_all_path = ARTIFACTS_DIR / "metrics_all_splits.json"
metrics_test_path = ARTIFACTS_DIR / "metrics_test.csv"
search_summary_path = ARTIFACTS_DIR / "search_summary.json"

results_df.to_json(metrics_all_path, orient="records", force_ascii=False, indent=2)
test_summary.to_csv(metrics_test_path, index=False)
search_summary_path.write_text(json.dumps(search_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:")
print("-", metrics_all_path.name)
print("-", metrics_test_path.name)
print("-", search_summary_path.name)

## 6. Лучшая модель: быстрая диагностика на тесте

После выбора лучшей модели смотрим на:

- confusion matrix;
- ROC-кривую;
- permutation importance.

Важно: мы **не меняем модель после просмотра теста**.

In [ ]:
best_row = select_best_row(results_df)
best_model_name = best_row["model"]
best_model = model_store[best_model_name]

print("Best model:", best_model_name)
print("Test ROC-AUC:", best_row["roc_auc"])
print("Test F1:", best_row["f1"])
print("Best params:", best_row["best_params"])

In [ ]:
y_test_pred = best_model.predict(X_test)

if hasattr(best_model, "predict_proba"):
    y_test_score = best_model.predict_proba(X_test)[:, 1]
elif hasattr(best_model, "decision_function"):
    y_test_score = best_model.decision_function(X_test)
else:
    y_test_score = None

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion matrix: {best_model_name}")

if y_test_score is not None:
    RocCurveDisplay.from_predictions(y_test, y_test_score, ax=axes[1])
    axes[1].set_title(f"ROC curve: {best_model_name}")
else:
    axes[1].text(0.5, 0.5, "ROC недоступна", ha="center", va="center", fontsize=12)
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    n_repeats=15,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values(by="importance_mean", ascending=False).reset_index(drop=True)

top_k = 15
display(importance_df.head(top_k))

plt.figure(figsize=(8, 6))
plt.barh(
    importance_df.head(top_k)["feature"][::-1],
    importance_df.head(top_k)["importance_mean"][::-1],
)
plt.xlabel("Permutation importance (mean ROC-AUC drop)")
plt.title(f"Top-{top_k} feature importances: {best_model_name}")
plt.tight_layout()
plt.show()

## 7. Сохранение лучшей модели

Сохраним лучшую модель и небольшой JSON-паспорт результата.

In [ ]:
best_model_path = ARTIFACTS_DIR / "best_model.joblib"
best_model_meta_path = ARTIFACTS_DIR / "best_model_meta.json"
importance_path = ARTIFACTS_DIR / "best_model_permutation_importance.csv"

joblib.dump(best_model, best_model_path)

best_model_meta = {
    "best_model_name": best_model_name,
    "selection_rule": "max test roc_auc, then f1, then accuracy",
    "test_metrics": {
        "accuracy": None if pd.isna(best_row["accuracy"]) else float(best_row["accuracy"]),
        "precision": None if pd.isna(best_row["precision"]) else float(best_row["precision"]),
        "recall": None if pd.isna(best_row["recall"]) else float(best_row["recall"]),
        "f1": None if pd.isna(best_row["f1"]) else float(best_row["f1"]),
        "roc_auc": None if pd.isna(best_row["roc_auc"]) else float(best_row["roc_auc"]),
    },
    "best_cv_score": None if pd.isna(best_row["best_cv_score"]) else float(best_row["best_cv_score"]),
    "best_params": best_row["best_params"],
    "random_state": RANDOM_STATE,
    "train_shape": list(X_train.shape),
    "test_shape": list(X_test.shape),
}

best_model_meta_path.write_text(json.dumps(best_model_meta, ensure_ascii=False, indent=2), encoding="utf-8")
importance_df.to_csv(importance_path, index=False)

print("Saved:")
print("-", best_model_path.name)
print("-", best_model_meta_path.name)
print("-", importance_path.name)

## 8. Итоги

1. **Модели нужно сравнивать в одном и том же протоколе.**
2. **Подбор делается только на train.**
3. **Сравнение должно быть воспроизводимым.**
4. **Разные семейства моделей дают разный профиль качества.**
5. **Хороший эксперимент заканчивается артефактами.**

Именно это и есть основная идея S08:  
не просто «попробовать несколько моделей», а **аккуратно выбрать лучшую так, чтобы результат можно было защитить и повторить**.